# Laboratorio 3 — Detección de Anomalías en Tráfico de Red con Machine Learning

**Estudiante:** Cristian Cabana Sulca  
**Curso:** Seguridad Informática — Unidad IV  
**Dataset:** `lab3/network_traffic.csv` (10 000 registros, 30 días de tráfico de red)  
**Modelo:** Isolation Forest (no supervisado)

## Tarea 3.1 — Exploración y Preprocesamiento

In [ ]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import precision_score, recall_score, f1_score, confusion_matrix, classification_report
import joblib
import warnings
warnings.filterwarnings('ignore')

print('Librerias cargadas correctamente')

In [ ]:
# Carga del dataset
df = pd.read_csv('lab3/network_traffic.csv', parse_dates=['timestamp'])

print(f'Forma del dataset: {df.shape}')
print(f'\nColumnas: {list(df.columns)}')
print(f'\nDistribucion de etiquetas:')
print(df['label'].value_counts())
print(f'\nEstadisticas descriptivas:')
df.describe()

In [ ]:
# Histogramas de bytes_sent y duration_sec
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(df['bytes_sent'], bins=50, color='steelblue', edgecolor='black', alpha=0.8)
axes[0].set_title('Distribucion de bytes_sent')
axes[0].set_xlabel('Bytes enviados')
axes[0].set_ylabel('Frecuencia')

axes[1].hist(df['duration_sec'], bins=50, color='darkorange', edgecolor='black', alpha=0.8)
axes[1].set_title('Distribucion de duration_sec')
axes[1].set_xlabel('Duracion (segundos)')
axes[1].set_ylabel('Frecuencia')

plt.tight_layout()
plt.savefig('lab3/evidencias/SCR-3.1_eda.png', dpi=120, bbox_inches='tight')
plt.show()
print('Histogramas guardados')

In [ ]:
# Verificacion y tratamiento de valores nulos
print('Valores nulos por columna:')
print(df.isnull().sum())

# Eliminar filas con nulos si existen
df = df.dropna()
print(f'\nFilas tras eliminar nulos: {len(df)}')

# Tratamiento de outliers extremos con clipeo en percentil 99
for col in ['bytes_sent', 'bytes_recv', 'duration_sec', 'packets']:
    p99 = df[col].quantile(0.99)
    outliers = (df[col] > p99).sum()
    df[col] = df[col].clip(upper=p99)
    print(f'{col}: {outliers} valores clipados al percentil 99 ({p99:.2f})')

In [ ]:
# Feature Engineering: 2 nuevas variables derivadas

# ratio_bytes: relacion entre bytes enviados y recibidos (>1 = mas saliente que entrante)
df['ratio_bytes'] = df['bytes_sent'] / (df['bytes_recv'] + 1)

# bytes_por_segundo: tasa de transferencia de salida
df['bytes_por_segundo'] = df['bytes_sent'] / (df['duration_sec'] + 0.001)

print('Variables derivadas creadas:')
print(df[['ratio_bytes', 'bytes_por_segundo']].describe())

In [ ]:
# Normalizacion con StandardScaler
FEATURES = ['bytes_sent', 'bytes_recv', 'duration_sec', 'packets',
            'dst_port', 'ratio_bytes', 'bytes_por_segundo']

scaler = StandardScaler()
X_scaled = scaler.fit_transform(df[FEATURES])

print(f'Features normalizadas: {FEATURES}')
print(f'Media post-escala (debe ser ~0): {X_scaled.mean(axis=0).round(4)}')
print(f'Std post-escala (debe ser ~1):  {X_scaled.std(axis=0).round(4)}')

## Tarea 3.2 — Entrenamiento del Modelo Isolation Forest

In [ ]:
# Entrenamiento del modelo (sin usar la columna label)
modelo = IsolationForest(
    contamination=0.05,
    n_estimators=100,
    random_state=42
)
modelo.fit(X_scaled)

# Predicciones: -1 = anomalia, 1 = normal
predicciones = modelo.predict(X_scaled)

df['prediccion'] = predicciones
print(f'Anomalias detectadas: {(predicciones == -1).sum()}')
print(f'Normal:               {(predicciones == 1).sum()}')

In [ ]:
# Metricas de evaluacion usando la columna label como ground truth
# Conversion: anomaly -> -1, normal -> 1
y_true = df['label'].map({'anomaly': -1, 'normal': 1})
y_pred = df['prediccion']

# Para sklearn las metricas usan 0/1: anomaly=1 (positivo), normal=0
y_true_bin = (y_true == -1).astype(int)
y_pred_bin = (y_pred == -1).astype(int)

precision = precision_score(y_true_bin, y_pred_bin)
recall    = recall_score(y_true_bin, y_pred_bin)
f1        = f1_score(y_true_bin, y_pred_bin)

print('=== Metricas de Evaluacion ===')
print(f'Precision : {precision:.4f}')
print(f'Recall    : {recall:.4f}')
print(f'F1-Score  : {f1:.4f}')
print()
print(classification_report(y_true_bin, y_pred_bin,
      target_names=['Normal', 'Anomalia']))

In [ ]:
# Matriz de confusion con seaborn
cm = confusion_matrix(y_true_bin, y_pred_bin)

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Normal', 'Anomalia'],
            yticklabels=['Normal', 'Anomalia'], ax=ax)
ax.set_ylabel('Real')
ax.set_xlabel('Predicho')
ax.set_title('Matriz de Confusion — Isolation Forest')
plt.tight_layout()
plt.savefig('lab3/evidencias/SCR-3.2_metricas.png', dpi=120, bbox_inches='tight')
plt.show()
print('Matriz de confusion guardada')

## Tarea 3.3 — Interpretación y Umbral Dinámico

In [ ]:
# Score de anomalia: valores mas negativos = mas anomalos
scores = modelo.decision_function(X_scaled)
df['anomaly_score'] = scores

fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(range(len(scores)), sorted(scores), color='crimson', linewidth=0.5)
ax.axhline(y=0, color='black', linestyle='--', linewidth=1, label='Umbral por defecto (0)')
ax.set_xlabel('Registros (ordenados por score)')
ax.set_ylabel('Anomaly Score (decision_function)')
ax.set_title('Score de Anomalia para todos los registros')
ax.legend()
plt.tight_layout()
plt.show()
print(f'Score minimo (mas anomalo): {scores.min():.4f}')
print(f'Score maximo (mas normal):  {scores.max():.4f}')

In [ ]:
# Curva Umbral vs F1-Score para encontrar el umbral optimo
thresholds = np.linspace(scores.min(), scores.max(), 200)
f1_scores = []

for t in thresholds:
    y_pred_t = (scores < t).astype(int)  # anomalia si score < umbral
    f1_scores.append(f1_score(y_true_bin, y_pred_t, zero_division=0))

best_idx = np.argmax(f1_scores)
best_threshold = thresholds[best_idx]
best_f1 = f1_scores[best_idx]

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(thresholds, f1_scores, color='steelblue', linewidth=2)
ax.axvline(x=best_threshold, color='red', linestyle='--',
           label=f'Umbral optimo: {best_threshold:.4f} (F1={best_f1:.4f})')
ax.set_xlabel('Umbral (threshold)')
ax.set_ylabel('F1-Score')
ax.set_title('Curva Umbral vs F1-Score')
ax.legend()
plt.tight_layout()
plt.savefig('lab3/evidencias/SCR-3.3_umbral_f1.png', dpi=120, bbox_inches='tight')
plt.show()

print(f'Umbral optimo: {best_threshold:.4f}')
print(f'F1-Score maximo: {best_f1:.4f}')

In [ ]:
# Top 10 registros mas anomalos
top10_anomalos = df.nsmallest(10, 'anomaly_score')
print('Top 10 registros mas anomalos:')
top10_anomalos[['timestamp','src_ip','dst_ip','dst_port','protocol',
                'bytes_sent','bytes_recv','duration_sec','packets',
                'label','anomaly_score']]

### Analisis del Top 10 de Registros mas Anomalos

Los registros con score de anomalia mas bajo (mas negativos) se destacan por las siguientes caracteristicas que los convierten en posibles amenazas reales:

1. **Volumen de datos inusualmente alto (`bytes_sent`):** Transferencias que superan ampliamente el promedio del dataset pueden indicar **exfiltracion de datos**. Un host interno enviando cientos de MB hacia una IP externa no conocida es una señal critica.

2. **`ratio_bytes` elevado (mucho mas enviado que recibido):** En comunicaciones normales cliente-servidor, el cliente recibe mas de lo que envia. Un ratio muy alto invierte esta logica y sugiere que el host es la fuente de la exfiltracion, no el receptor.

3. **`bytes_por_segundo` extremo:** Una tasa de transferencia muy alta en poco tiempo (`duration_sec` bajo con `bytes_sent` alto) puede indicar un **volcado masivo de informacion** o una herramienta automatizada de exfiltracion.

4. **Puertos de destino inusuales (`dst_port`):** Conexiones a puertos no estandar (fuera del rango 80, 443, 22) desde hosts internos hacia IPs externas pueden indicar **comunicacion con C2 (Command & Control)** o tunel de datos encubierto.

5. **Protocolo ICMP con datos:** El protocolo ICMP no deberia transportar grandes volumenes de datos. Si aparece con `bytes_sent` alto, puede indicar **ICMP tunneling**, una tecnica de exfiltracion encubierta.

Estos patrones combinados representan indicadores tipicos de compromiso (IOC) en un entorno empresarial y deben correlacionarse con alertas del SIEM para confirmar un incidente.

## Tarea 3.4 — Exportacion del Modelo

In [ ]:
# Serializar modelo y scaler con joblib
import os
os.makedirs('lab3/evidencias', exist_ok=True)

joblib.dump(modelo, 'lab3/modelo_anomalias.pkl')
joblib.dump(scaler, 'lab3/scaler.pkl')

print('Modelo serializado en: lab3/modelo_anomalias.pkl')
print('Scaler serializado en: lab3/scaler.pkl')